# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

You will learn how to use unique Croissant `@id` identifiers to interact with all dataset entities including record sets and fields.

In [ ]:
# Ensure that mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata as an object (not dictionary)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
Review available record sets (tables) and fields referencing their Croissant `@id` fields.

In [ ]:
# List all available record sets and their fields by @id

print("Available record sets and their fields (using @id):\n")
record_set_ids = []
for rs in dataset.record_sets:
    # Each record set is a Croissant RecordSet object
    print(f"- Record set name: {rs.name}")
    print(f"  @id: {rs.id}")
    field_ids = [f.id for f in rs.fields]
    print(f"  Fields (@id): {field_ids}\n")
    record_set_ids.append(rs.id)

# Optionally print record set names for quick reference
print("Collected record set @id values:")
print(record_set_ids)

Let's sample a few records from every record set using their `@id`.

In [ ]:
# Inspect a few records from each record set, referencing by @id
for record_set_id in record_set_ids:
    print(f"\nFirst 2 records from record set {record_set_id}:")
    for i, record in enumerate(dataset.records(record_set=record_set_id)):
        print(record)
        if i >= 1:
            break

## 3. Data Extraction
We will extract all records from each record set into a dictionary of DataFrames, using the record set `@id` as the key.

You can select fields or manipulate columns by referencing their Croissant `@id` directly.

In [ ]:
# Load all data into DataFrames using @id as the key for each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for record set {record_set_id}: {df.columns.tolist()}")
    print(df.head(2), "\n")

# We'll select the first record set for in-depth analysis (by @id)
main_record_set_id = record_set_ids[0]
print(f"Using record set for EDA: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply simple processing: filtering, normalization and grouping using numeric and categorical fields, **referenced by their `@id`**.

In [ ]:
# We'll attempt to identify a numeric field (e.g., age) and a group field (e.g., sex/comorbidity) by @id
# List columns for main record set
main_df = dataframes[main_record_set_id]
print("Available columns in main record set:")
print(main_df.columns.tolist())

# For illustration, select numeric_field_id and group_field_id by reviewing the columns
# (Replace these values with the actual @id from your printed column list above)
numeric_field_id = None
group_field_id = None

# We'll try to auto-detect a plausible numeric field and group field by name heuristics
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col

# If age not found, fallback to first float/int column
if numeric_field_id is None:
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field_id = col
            break
if group_field_id is None:
    for col in main_df.columns:
        if main_df[col].dtype == object and ('location' in col.lower() or 'site' in col.lower()):
            group_field_id = col
            break

print(f"\nSelected numeric field (@id): {numeric_field_id}")
print(f"Selected group field (@id): {group_field_id}")

if numeric_field_id is not None:
    # Use a threshold for filtering (e.g., age > 50)
    # If not age, filter on median+10
    field_values = main_df[numeric_field_id]
    if pd.api.types.is_numeric_dtype(field_values):
        threshold = min(50, field_values.median() + 10)
        filtered_df = main_df[field_values > threshold].copy()
        print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the selected numeric field
        mu, sigma = filtered_df[numeric_field_id].mean(), filtered_df[numeric_field_id].std()
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - mu) / sigma
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Grouped means by group_field (e.g. by sex or location)
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No numeric field found in main record set for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and, if possible, the group distribution as well.

All variable references should use their Croissant `@id` identifiers for reproducibility.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and pd.api.types.is_numeric_dtype(main_df[numeric_field_id]):
    plt.figure(figsize=(7,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id is not None and group_field_id in main_df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30)
        plt.show()
else:
    print("No numeric field found or selected for visualization.")

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to read and process the FAIR² dataset package described by a Croissant schema.

- All data entities are referenced by their Croissant `@id` for robust and reproducible workflows.
- You learned to load records, explore the available structure and fields, extract tables, and perform basic EDA, filtering, normalization, grouping, and visualizations.

You can now build on this foundation to develop advanced analyses using this or other Croissant-compatible datasets.

**Remember:** Always use the `@id` field to reference entities within the Croissant ecosystem for clarity and data lineage.